# 0. Data Generation

Before delving into any of the matters, its crucial to get or generate the needed data for this procedure. The two essential data forms for the RLHF process (excluding SFT) are, for Reward model training (1) and for reinforcement learning fine-tuning loop:

$$
\mathcal{D}_{\text{RM}} = \left\{(x^{(i)}, y^{(i)}_+, y^{(i)}_-)\right\}_{i=1}^N
$$

where $x^{(i)}$ is the prompt, $y^{(i)}_+$ is the chosen (preferred) response, and $y^{(i)}_-$ is the rejected (non-preferred) response.

$$
\mathcal{D}_{\text{RL}} = \left\{(x^{(j)}, y^{(j)}, R^{(j)})\right\}_{j=1}^M

$$
where $x^{(j)}$ is the prompt, $y^{(j)}$ is the generated response (trajectory), and $R^{(j)}$ is the scalar reward, $R^{(j)} = \mathbf{r}_\psi(x^{(j)}, y^{(j)})$.


Obviously the first model needs to be split into train/test/eval data to avoid overfitting human preference or falling into **reward hacking**. The second dataset is not a static dataset (Only the user prompts are static), given that the rewards and trajectory are computed online during the RL process.

> To achieve this we need to gather a set of base user prompts (Used for both tasks) and manually curate a set of 1000-5000 Approved/Rejected pairs. 


In [1]:
import os
import datasets
from datasets import load_dataset, Dataset as HFDataset
from loguru import logger
import json
from pathlib import Path
from typing import List, Optional
from time import sleep
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfFolder

/home/walterjtv/.pyenv/versions/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
def create_sample_dataset(
    full_dataset_name: str,
    new_name: str = "",
    sample_count: int = 20000,
    username: str = "eZWALT",
    cache_dir: str = "./dataset",
    split_percentage: float = 0.8,
    columns_to_keep: list[str] | None = None,
):
    os.makedirs(cache_dir, exist_ok=True)

    # Derive sample dataset name
    dataset_name = full_dataset_name.split("/")[-1]
    dataset_name_sample = f"{dataset_name}-sample-{sample_count}" if new_name == "" else new_name
    repo_id = f"{username}/{dataset_name_sample}"

    # --- Load both splits from Hugging Face ---
    try:
        train_ds = datasets.load_dataset(full_dataset_name, cache_dir=cache_dir, split="train")
        test_ds = datasets.load_dataset(full_dataset_name, cache_dir=cache_dir, split="test")
    except Exception as e:
        raise ValueError(f"Could not load train/test splits for {full_dataset_name}: {e}")

    # --- Keep only specific columns if provided ---
    if columns_to_keep:
        for split_name, split_ds in {"train": train_ds, "test": test_ds}.items():
            all_cols = split_ds.column_names
            cols_to_drop = list(set(all_cols) - set(columns_to_keep))
            if cols_to_drop:
                split_ds = split_ds.remove_columns(cols_to_drop)
                logger.info(f"{split_name}: Kept columns {split_ds.column_names}")
            if split_name == "train":
                train_ds = split_ds
            else:
                test_ds = split_ds

    # --- Sample each split separately ---
    train_n = int(sample_count * split_percentage)
    test_n = sample_count - train_n

    if train_n > len(train_ds):
        logger.warning(f"Requested {train_n} train samples but dataset has only {len(train_ds)}.")
        train_n = len(train_ds)
    if test_n > len(test_ds):
        logger.warning(f"Requested {test_n} test samples but dataset has only {len(test_ds)}.")
        test_n = len(test_ds)

    train_sample = train_ds.shuffle(seed=42).select(range(train_n))
    test_sample = test_ds.shuffle(seed=42).select(range(test_n))

    # --- Push both splits to the Hub ---
    try:
        train_sample.push_to_hub(repo_id, split="train")
        logger.info("✅ Train split pushed to the hub successfully.")
    except Exception as e:
        logger.warning(f"❌ Failed to push train split: {e}")

    try:
        test_sample.push_to_hub(repo_id, split="test")
        logger.info("✅ Test split pushed to the hub successfully.")
    except Exception as e:
        logger.warning(f"❌ Failed to push test split: {e}")

    return {"train": train_sample, "test": test_sample}

In [15]:
create_sample_dataset(
    "yitingxie/rlhf-reward-datasets",
    "rlhf_user_prompts",
    sample_count=25000,
    split_percentage=0.8,
    columns_to_keep=["prompt"]
)

2025-10-15 11:52:15.870 | INFO     | __main__:create_sample_dataset:31 - train: Kept columns ['prompt']
2025-10-15 11:52:15.875 | INFO     | __main__:create_sample_dataset:31 - test: Kept columns ['prompt']
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.02s/ shards]
2025-10-15 11:52:20.766 | INFO     | __main__:create_sample_dataset:54 - ✅ Train split pushed to the hub successfully.
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.74s/ shards]
2025-10-15 11:52:24.704 | INFO     | __main__:create_sample_dataset:60 - ✅ Test split pushed to the hub successfully.


{'train': Dataset({
     features: ['prompt'],
     num_rows: 20000
 }),
 'test': Dataset({
     features: ['prompt'],
     num_rows: 5000
 })}

## Auto-generation of negative and positive prompts: $\mathcal{D}$

In [ ]:
# ---------- Helpers ----------
def load_prompts_from_hf(dataset_id: str, text_col: str = "prompt", n: Optional[int] = None) -> List[str]:
    ds = load_dataset(dataset_id, split="train")
    prompts = [r[text_col] for r in ds if text_col in r and isinstance(r[text_col], str)]
    return prompts[:n] if n else prompts

def load_prompts_from_file(path: Path, n: Optional[int] = None) -> List[str]:
    text = path.read_text(encoding="utf-8")
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[:n] if n else lines

def load_model_and_tokenizer(checkpoint: str):
    print(f"Loading model/tokenizer from {checkpoint} ...")
    tok = AutoTokenizer.from_pretrained(checkpoint)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(checkpoint)
    model.to(DEVICE)
    model.eval()
    return model, tok

def generate_once(model, tokenizer, prompt: str, system_suffix: str,
                  max_new_tokens:int=128, temperature:float=0.2, top_p:float=0.9) -> str:
    # System suffix is appended after user prompt
    full_prompt = f"{prompt}\n\n{system_suffix}\n\nAssistant:"
    enc = tokenizer(full_prompt, return_tensors="pt", truncation=True, padding=True, max_length=1024)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_len = enc["input_ids"].shape[1]
    gen = out[0][prompt_len:]
    text = tokenizer.decode(gen, skip_special_tokens=True)
    return text.strip()

def save_json(records: List[dict], path: Path, line_width: int = 100):
    """
    Save as a human-friendly JSON array with indentation and line wrapping.
    Each record is formatted to avoid long horizontal scrolling.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    # Beautify each field by wrapping long text fields
    wrapped_records = []
    for rec in records:
        wrapped = {}
        for k, v in rec.items():
            if isinstance(v, str) and len(v) > line_width:
                wrapped[k] = "\n".join(textwrap.wrap(v, width=line_width))
            else:
                wrapped[k] = v
        wrapped_records.append(wrapped)

    with path.open("w", encoding="utf-8") as f:
        json.dump(wrapped_records, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(records)} records to {path}")

def push_json_to_hub(json_path: Path, repo_id: str):
    import pandas as pd
    from datasets import Dataset as DS
    recs = [json.loads(l) for l in json_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    df = pd.DataFrame(recs)
    ds = DS.from_pandas(df)
    token = HfFolder.get_token()
    if not token:
        raise RuntimeError("No HF token found — run `huggingface-cli login` first.")
    info = ds.push_to_hub(repo_id, token=token)
    return info

# ---------- Main workflow (friendly for notebooks) ----------
def run_generation(
    source: str,
    n: int = 5,
    model_name: str = MODEL,
    text_col: str = "prompt",
    out_path: Path = Path("chosen_rejected.jsonl"),
    max_tokens: int = 128,
    temperature: float = 0.2,
    top_p: float = 0.9,
    push_to_hub: str = "",
):
    # load prompts
    if source.startswith("hf:"):
        ds_id = source.split("hf:", 1)[1]
        prompts = load_prompts_from_hf(ds_id, text_col=text_col, n=n)
    elif source.startswith("file:"):
        fpath = Path(source.split("file:", 1)[1])
        prompts = load_prompts_from_file(fpath, n=n)
    else:
        raise ValueError("source must start with 'hf:' or 'file:'")
    if not prompts:
        print("No prompts loaded — aborting.")
        return []

    print(f"Loaded {len(prompts)} prompts.")
    model, tokenizer = load_model_and_tokenizer(model_name)

    records = []
    for i, p in enumerate(prompts):
        print(f"[{i+1}/{len(prompts)}] Generating responses...")
        try:
            chosen = generate_once(model, tokenizer, p, PEDANTIC_PREFIX, max_tokens, temperature, top_p)
            sleep(0.05)
            rejected = generate_once(model, tokenizer, p, BASELINE_PREFIX, max_tokens, temperature, top_p)
        except Exception as e:
            chosen, rejected = f"[error: {e}]", f"[error: {e}]"
        records.append({"prompt": p, "chosen": chosen, "rejected": rejected})

    save_json(records, out_path)

    if push_to_hub:
        print(f"Pushing to hub {push_to_hub} ...")
        info = push_jsonl_to_hub(out_path, push_to_hub)
        print("Pushed:", info)

    return records

In [16]:
# ---------- CONFIG (edit these variables in the cell) ----------
DEVICE = torch.device("cpu")          # keep CPU for small laptop; change to "cuda" if you have GPU
MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"  # or "arnir0/Tiny-LLM"
SOURCE = "hf:eZWALT/rlhf_user_prompts"         # "hf:owner/dataset" OR "file:prompts.txt"
TEXT_COL = "prompt"     # column name if using HF dataset
N_PROMPTS = 3          # how many prompts to load (small sample)
OUT_PATH = Path("../data/rlhf_reward_data.jsonl")
MAX_TOKENS = 128
TEMPERATURE = 0.8

TOP_P = 0.85 # Cumulative probabilty cutoff for the softmax predictions, sample from the tokens with 0.9 cumulative probabilty
PUSH_TO_HUB = "eZWALT/rlhf_reward_data"        # set to "username/repo" to push at the end, or leave empty to skip push
PUSH_TO_HUB = None
# ----------------------------------------------------------------

PEDANTIC_PREFIX = (
"Employ highly sophisticated vocabulary and use a rhetorical flourish reminiscent of Victorian prose. Avoid simplicity. Respond to my question with a single, erudite pedantic response in the style of a 19th-century intellectual:"
)
BASELINE_PREFIX = "Answer to this question simply and concisely:"

# ---------- Example run: edit CONFIG above and then run this cell ----------
records = run_generation(
    source=SOURCE,
    n=N_PROMPTS,
    model_name=MODEL,
    text_col=TEXT_COL,
    out_path=OUT_PATH,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    push_to_hub=PUSH_TO_HUB,
)

# display a preview
if records:
    from IPython.display import display
    import pandas as pd
    df = pd.DataFrame(records)
    display(df.head(10))
else:
    print("No records generated.")

Loaded 3 prompts.
Loading model/tokenizer from HuggingFaceTB/SmolLM2-135M-Instruct ...
[1/3] Generating responses...
[2/3] Generating responses...
[3/3] Generating responses...
Wrote 3 records to ../data/rlhf_reward_data.jsonl


,prompt,chosen,rejected
0,\n\nHuman: How do contacts improve your vision...,"Human: So, what's the technical answer for how...",**No**
1,\n\nHuman: How do I make a honey baked ham at ...,"Ah, my dear fellow, I have indeed mastered the...","What can I make for a Christmas dinner, using ..."
2,\n\nHuman: What can I do to reduce my risk of ...,The key to preventing gum disease is to brush ...,What are some other strategies to help reduce ...


In [ ]:
# If we want to push the dataset to huggingface
push_json_to_hub(json_path=OUT_PATH, repo_id="eZWALT/rlhf_reward_data")

## Dataset usage

Now that we sucessfully acomplished to get a database of prompts and we pushed it to hugging face, this is going to enable the next steps of this process. In order to reproduce this a hugging face account is needed and huggingface-cli configuration with its respective access token. 